## CUDA GEMM Benchmark (Colab)

This notebook assumes you copied the repo's `cuda-benchmark/` folder to the root of your Google Drive at:

`MyDrive/cuda-benchmark/`

It will:
- Mount Google Drive
- Compile `cublas_benchmark.cu` with `nvcc`
- Run the PyTorch benchmark
- Run the cuBLAS benchmark
- Run the plotting script (writes plots to `plots/`)


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# (Optional) sanity checks
!nvidia-smi
!nvcc --version


Fri Mar 20 03:03:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Go to cuda-benchmark folder in Drive
import os
bench_dir = '/content/drive/MyDrive/cuda-benchmark'
assert os.path.isdir(bench_dir), f"Missing {bench_dir}. Copy the repo's cuda-benchmark folder there."
%cd {bench_dir}
!ls -la


/content/drive/MyDrive/cuda-benchmark
total 29
-rw------- 1 root root 12872 Mar 20 02:40 cublas_benchmark.cu
-rw------- 1 root root  3335 Mar 20 01:51 cuda_benchmark_colab.ipynb
-rw------- 1 root root  6674 Mar 20 02:50 plot_benchmarks.py
-rw------- 1 root root  4937 Mar 20 02:40 pytorch_benchmark.py


In [4]:
# Ensure plotting deps exist (PyTorch is typically preinstalled on Colab GPUs)
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())"
!pip -q install matplotlib


torch 2.10.0+cu128 cuda 12.8 available True


In [5]:
# Compile cuBLAS benchmark
# Note: Colab has CUDA + cuBLAS available on the system image
!nvcc -O3 --std=c++17 cublas_benchmark.cu -o cublas_benchmark -lcublas
!ls -la


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
total 1026
-rwx------ 1 root root 1018008 Mar 20 03:03 cublas_benchmark
-rw------- 1 root root   12872 Mar 20 02:40 cublas_benchmark.cu
-rw------- 1 root root    5845 Mar 20 03:03 cuda_benchmark_colab.ipynb
-rw------- 1 root root    6674 Mar 20 02:50 plot_benchmarks.py
-rw------- 1 root root    4937 Mar 20 02:40 pytorch_benchmark.py


In [6]:
# Run PyTorch benchmark using script defaults
!python pytorch_benchmark.py


Wrote 18 entries to results/pytorch_benchmark.json


In [7]:
# Run cuBLAS benchmark using binary defaults
!./cublas_benchmark


mode=cuda_core_fp32 hidden=64 avg_ms=0.033 TFLOPs=8.05
mode=cuda_core_fp32 hidden=128 avg_ms=0.057 TFLOPs=9.47
mode=cuda_core_fp32 hidden=256 avg_ms=0.096 TFLOPs=11.19
mode=cuda_core_fp32 hidden=512 avg_ms=0.160 TFLOPs=13.43
mode=cuda_core_fp32 hidden=1024 avg_ms=0.278 TFLOPs=15.45
mode=cuda_core_fp32 hidden=2048 avg_ms=0.494 TFLOPs=17.39
mode=cuda_core_fp32 hidden=4096 avg_ms=0.950 TFLOPs=18.09
mode=cuda_core_fp32 hidden=8192 avg_ms=1.998 TFLOPs=17.20
mode=cuda_core_fp32 hidden=16384 avg_ms=3.957 TFLOPs=17.36
mode=tensor_core_tf32 hidden=64 avg_ms=0.023 TFLOPs=11.53
mode=tensor_core_tf32 hidden=128 avg_ms=0.026 TFLOPs=20.38
mode=tensor_core_tf32 hidden=256 avg_ms=0.033 TFLOPs=32.43
mode=tensor_core_tf32 hidden=512 avg_ms=0.065 TFLOPs=32.94
mode=tensor_core_tf32 hidden=1024 avg_ms=0.081 TFLOPs=52.87
mode=tensor_core_tf32 hidden=2048 avg_ms=0.121 TFLOPs=70.70
mode=tensor_core_tf32 hidden=4096 avg_ms=0.181 TFLOPs=94.86
mode=tensor_core_tf32 hidden=8192 avg_ms=0.319 TFLOPs=107.81
mode=ten

In [8]:
# Plot comparison curves using script defaults
!python plot_benchmarks.py
!ls -la plots


Saved plots to plots
total 325
-rw------- 1 root root 69794 Mar 20 03:03 combined_latency_vs_hidden.png
-rw------- 1 root root 70587 Mar 20 03:03 combined_tflops_vs_hidden.png
-rw------- 1 root root 51071 Mar 20 03:03 cublas_latency_vs_hidden.png
-rw------- 1 root root 45424 Mar 20 03:03 cublas_tflops_vs_hidden.png
-rw------- 1 root root 47156 Mar 20 03:03 pytorch_latency_vs_hidden.png
-rw------- 1 root root 46785 Mar 20 03:03 pytorch_tflops_vs_hidden.png
